# 04 Очистка данных о звонках (Calls)

В этом блокноте мы проведем очистку данных о звонках из файла `Calls (Done).xlsx` и подготовим их для анализа эффективности воронки продаж.

In [5]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import re

import help_130625_dam as h

# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Загрузка данных

In [10]:
DATA_PATH = os.path.join('..', 'Sources', 'Calls (Done).xlsx')

# Читаем ID как строки (dtype=str), это предотвращает округление 19-значных чисел при загрузке
df = pd.read_excel(DATA_PATH, dtype={'Id': str, 'CONTACTID': str})

# Нормализация имён столбцов в snake_case
def _to_snake(col):
    col = col.lower()
    col = re.sub(r'[\s\(\)]+', '_', col)
    return col.strip('_')

df.columns = [_to_snake(c) for c in df.columns]

print(f"Загружено строк: {len(df)}")
h.descr_df(df, include=['number', 'object'], show_sample_rows=True)

Загружено строк: 95874


,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3,Минимум,Среднее,Медиана,Максимум
0,id,str,95874,0,95874,5805028000000805001,5805028000000768006,5805028000000764027,NaN,NaN,NaN,NaN
1,call_start_time,str,95874,0,68445,30.06.2023 08:43,30.06.2023 08:46,30.06.2023 08:59,NaN,NaN,NaN,NaN
2,call_owner_name,str,95874,0,33,John Doe,John Doe,John Doe,NaN,NaN,NaN,NaN
3,contactid,str,91941,3933,15214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,call_type,str,95874,0,3,Inbound,Outbound,Outbound,NaN,NaN,NaN,NaN
5,call_duration_in_seconds,float64,95791,83,2619,171.00,28.00,24.00,0.00,164.98,8.00,7625.00
6,call_status,str,95874,0,11,Received,Attended Dialled,Attended Dialled,NaN,NaN,NaN,NaN
7,dialled_number,float64,0,95874,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,outgoing_call_status,str,86875,8999,4,NaN,Completed,Completed,NaN,NaN,NaN,NaN
9,scheduled_in_crm,float64,86875,8999,2,NaN,0.00,0.00,0.00,0.00,0.00,1.00


## Первичный анализ и исправление типов

In [11]:
# 1. Исправление ID типов (предотвращение потери точности)

# id: pd.to_numeric безопасен — id не содержит NaN
df['id'] = pd.to_numeric(df['id'], errors='coerce').astype('Int64')

# contactid: используем pd.to_numeric с последующим приведением к Int64.
df['contactid'] = pd.to_numeric(df['contactid'], errors='coerce').astype('Int64')

# 2. Исправление дат
date_cols = [col for col in df.columns if 'time' in col or 'date' in col]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)

# 3. Базовая очистка и оптимизация типов
df['call_duration_in_seconds'] = df['call_duration_in_seconds'].fillna(0).astype('int32')
# Преобразуем в bool: 1 -> True, 0 -> False
df['scheduled_in_crm'] = df['scheduled_in_crm'].fillna(0).astype(bool)

# Текстовые столбцы -> category (для оптимизации памяти)
str_cols = ['call_owner_name', 'call_type', 'call_status']
for col in str_cols:
    df[col] = df[col].astype('category')

# 4. Удаление неиспользуемых столбцов
cols_to_drop = ['dialled_number', 'tag', 'outgoing_call_status']
df = df.drop(columns=cols_to_drop, errors='ignore')

print(f'contactid dtype: {df["contactid"].dtype} | NaN: {df["contactid"].isna().sum()}')
print(f'Пример contactid: {df["contactid"].dropna().iloc[0] if not df["contactid"].dropna().empty else "None"}')
df.info()

contactid dtype: Int64 | NaN: 3933
Пример contactid: 5805028000000645120
<class 'pandas.DataFrame'>
RangeIndex: 95874 entries, 0 to 95873
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   id                        95874 non-null  Int64         
 1   call_start_time           95874 non-null  datetime64[us]
 2   call_owner_name           95874 non-null  category      
 3   contactid                 91941 non-null  Int64         
 4   call_type                 95874 non-null  category      
 5   call_duration_in_seconds  95874 non-null  int32         
 6   call_status               95874 non-null  category      
 7   scheduled_in_crm          95874 non-null  bool          
dtypes: Int64(2), bool(1), category(3), datetime64[us](1), int32(1)
memory usage: 3.1 MB


In [8]:
# Проверка на перекрывающиеся звонки у одного менеджера
# Рассчитаем время окончания звонка
df['call_end_time'] = df['call_start_time'] + pd.to_timedelta(df['call_duration_in_seconds'], unit='s')

# Сортируем для проверки перекрытий
df_sorted = df.sort_values(['call_owner_name', 'call_start_time'])

# Сдвигаем время окончания предыдущего звонка того же менеджера
df_sorted['prev_call_end'] = df_sorted.groupby('call_owner_name')['call_end_time'].shift(1)

# Условие перекрытия
overlapping = df_sorted[df_sorted['call_start_time'] < df_sorted['prev_call_end']].copy()

if len(overlapping) > 0:
    print(f"Обнаружено {len(overlapping)} строк с пересечением времени.")
    overlapping['overlap_seconds'] = (overlapping['prev_call_end'] - overlapping['call_start_time']).dt.total_seconds()
    print("\nРаспределение величины пересечения (секунды):")
    print(overlapping['overlap_seconds'].describe())
else:
    print("Пересекающихся звонков не обнаружено.")


Обнаружено 9107 строк с пересечением времени.

Распределение величины пересечения (секунды):
count   9107.00
mean      67.86
std      258.12
min        1.00
25%        5.00
50%        7.00
75%       13.00
max     7140.00
Name: overlap_seconds, dtype: float64


### Выводы по пересекающимся звонкам:
В данных обнаружено **9107** случаев временного перекрытия звонков у одного и того же менеджера.

**Возможные причины:**
1. **Технические особенности CRM (75% случаев):** Большинство накладок составляют менее 13 секунд. Это может быть связано с тем, что система начинает запись нового звонка или автодозвон до того, как менеджер закроет карточку предыдущего клиента.
2. **Параллельные линии:** Использование гарнитур с поддержкой нескольких вызовов или работа в нескольких вкладках CRM одновременно.
3. Система может инициировать звонок заранее, чтобы минимизировать простой менеджера.
4. **Ошибки логирования данных:** Неточное фиксирование времени завершения (`Call End Time`) при обрыве связи или программных сбоях.
5. **Аномалии (длинные пересечения):** Одиночные случаи накладок в несколько десятков минут могут указывать на "зависшие" сессии звонков, которые не были корректно завершены в системе.

*Данные аномалии не критичны для общего анализа воронки, так как составляют менее 10% данных и в большинстве своем являются короткими техническими накладками.*


In [9]:
h.descr_df(df, include=['int', 'category', 'datetime'], show_sample_rows=False)

,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Минимум,Среднее,Медиана,Максимум
0,id,Int64,95874,0,95874,5805028000000764027,5805028000031678464.00,5805028000032743424.00,5805028000056912329
1,call_start_time,datetime64[us],95874,0,68445,<NA>,<NA>,<NA>,<NA>
2,call_owner_name,category,95874,0,33,<NA>,<NA>,<NA>,<NA>
3,contactid,Int64,91941,3933,7447,5805028000000645120,5805028000026525696.00,5805028000025716736.00,5805028000056892416
4,call_type,category,95874,0,3,<NA>,<NA>,<NA>,<NA>
5,call_duration_in_seconds,int32,95874,0,2619,0,164.83,8.00,7625
6,call_status,category,95874,0,11,<NA>,<NA>,<NA>,<NA>
7,call_end_time,datetime64[us],95874,0,90604,<NA>,<NA>,<NA>,<NA>


In [9]:
# Сохранение очищенных данных
CLEAN_DATA_PATH = os.path.join('..', 'data', 'cleaned', 'calls_clean.pkl')

os.makedirs(os.path.dirname(CLEAN_DATA_PATH), exist_ok=True)

df.to_pickle(CLEAN_DATA_PATH)
print(f"Файл сохранен: {CLEAN_DATA_PATH}")
print(f"Финальное количество строк: {len(df)}")
print(f"Успешных звонков: {df['is_successful'].sum()}")


Файл сохранен: ..\data\cleaned\calls_clean.pkl
Финальное количество строк: 95874
Успешных звонков: 73816


## Описание датасета

**Источник:** `Calls.xlsx` — выгрузка данных о звонках из CRM  
**Назначение:** анализ активности менеджеров, оценка качества обработки лидов и расчет метрик дозвона

### Ключевые аналитические показатели
| Столбец | Тип | Описание |
|---|---|---|
| `id` | `int64` | Уникальный ID записи о звонке |
| `contactid` | `int64` | ID связанного контакта (связь с `contacts.id`) |
| `call_start_time` | `datetime` | Дата и время начала звонка |
| `call_duration_in_seconds` | `int32` | Длительность разговора в секундах |
| `is_successful` | `bool` | **Флаг дозвона:** True, если длительность > 0 и статус успешный |
| `call_owner_name` | `category` | Менеджер, совершивший или принявший звонок |
| `call_type` | `category` | Тип звонка (Inbound/Outbound) |
| `call_status` | `category` | Результат (Completed, Missed и др.) |

### Особенности данных
- **Пропуски в `contactid`:** ~4% записей не имеют ID контакта (например, входящие с неизвестных номеров или ошибки логирования). Пропуски **не заполняются**, так как ID является уникальным внешним ключом. При объединении такие звонки не будут привязаны к воронке продаж, но сохранятся для анализа общей нагрузки менеджеров.
- **Пересекающиеся звонки:** в данных присутствуют технические накладки (менее 10%), вызванные особенностями работы CRM и телефонии.

**Ключевые связи:**
- `contactid` → [01_cleaning_contacts.ipynb](01_cleaning_contacts.ipynb) (`id`)
- `call_start_time` → расчет времени до первого контакта (Speed to Lead)
- `is_successful` → фильтр для анализа только результативных переговоров